In [ ]:
# data load

from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()

if not (project_root / "data" / "train.csv").exists():
    project_root = project_root.parent

data_dir = project_root / "data"

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

date_format = "%m/%d/%Y %I:%M:%S %p"

missing_drop_threshold = 0.90
max_onehot_cardinality = 15
unknown_value = "UNKNOWN"

print("train shape:", train.shape)
print("test shape:", test.shape)

In [ ]:
# 2) target, temporal features, leakage removal

def add_target(df):
    df = df.copy()

    created = pd.to_datetime(
        df["Created Date"],
        format=date_format,
        errors="coerce"
    )

    closed = pd.to_datetime(
        df["Closed Date"],
        format=date_format,
        errors="coerce"
    )

    hours_to_close = (closed - created).dt.total_seconds() / 3600

    df["target"] = (
        (hours_to_close >= 0) &
        (hours_to_close <= 24)
    ).astype(int)

    return df


def add_temporal_features(df):
    df = df.copy()

    created = pd.to_datetime(
        df["Created Date"],
        format=date_format,
        errors="coerce"
    )

    hour = created.dt.hour

    df["created_hour"] = hour
    df["created_day_of_week"] = created.dt.dayofweek
    df["created_month"] = created.dt.month
    df["created_is_weekend"] = created.dt.dayofweek.isin([5, 6]).astype(int)

    df["created_part_of_day"] = np.select(
        [
            hour.between(0, 5),
            hour.between(6, 11),
            hour.between(12, 17),
            hour.between(18, 23)
        ],
        [
            "night",
            "morning",
            "afternoon",
            "evening"
        ],
        default=unknown_value
    )

    return df


def convert_known_numeric_columns(df):
    df = df.copy()

    numeric_cols = [
        "Incident Zip",
        "Council District",
        "BBL",
        "X Coordinate (State Plane)",
        "Y Coordinate (State Plane)",
        "Latitude",
        "Longitude"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(",", ".", regex=False)
                .str.strip()
            )

            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def remove_leakage_and_ids(df):
    df = df.copy()

    cols_to_drop = [
        "Closed Date",
        "Created Date",
        "Unique Key",
        "Unnamed: 0",
        "Location"
    ]

    existing_cols = [
        col for col in cols_to_drop
        if col in df.columns
    ]

    return df.drop(columns=existing_cols)


train_processed = add_target(train)
train_processed = add_temporal_features(train_processed)
train_processed = convert_known_numeric_columns(train_processed)
train_processed = remove_leakage_and_ids(train_processed)

test_processed = add_temporal_features(test)
test_processed = convert_known_numeric_columns(test_processed)
test_processed = remove_leakage_and_ids(test_processed)

print(train_processed["target"].value_counts())
print(train_processed["target"].value_counts(normalize=True).round(4))

print("train processed shape:", train_processed.shape)
print("test processed shape:", test_processed.shape)

In [ ]:
# 3) missing values

def fit_missing_value_params(df):
    missing_fraction = df.isna().mean()

    dropped_cols = [
        col for col in missing_fraction[missing_fraction > missing_drop_threshold].index
        if col != "target"
    ]

    remaining = df.drop(columns=dropped_cols, errors="ignore")

    numeric_data = (
        remaining
        .select_dtypes(include="number")
        .drop(columns=["target"], errors="ignore")
    )

    numeric_medians = numeric_data.median().to_dict()

    return {
        "dropped_cols": dropped_cols,
        "numeric_medians": numeric_medians,
        "categorical_fill_value": unknown_value
    }


def apply_missing_value_params(df, params):
    df = df.copy()

    df = df.drop(columns=params["dropped_cols"], errors="ignore")

    for col, median in params["numeric_medians"].items():
        if col in df.columns:
            df[col] = df[col].fillna(median)

    categorical_cols = df.select_dtypes(include=["object", "category"]).columns

    for col in categorical_cols:
        df[col] = (
            df[col]
            .fillna(params["categorical_fill_value"])
            .astype(str)
        )

    return df


missing_params = fit_missing_value_params(train_processed)

train_processed = apply_missing_value_params(train_processed, missing_params)
test_processed = apply_missing_value_params(test_processed, missing_params)

print("dropped columns:")
print(missing_params["dropped_cols"])

print("train missing values:", train_processed.isna().sum().sum())
print("test missing values:", test_processed.isna().sum().sum())

In [ ]:
# 4) categorical encoding

def fit_categorical_encoding_params(df):
    categorical_cols = (
        df
        .select_dtypes(include=["object", "category"])
        .columns
        .tolist()
    )

    categorical_cols = [
        col for col in categorical_cols
        if col != "target"
    ]

    onehot_cols = []
    label_maps = {}

    for col in categorical_cols:
        n_unique = df[col].nunique(dropna=False)

        if n_unique <= max_onehot_cardinality:
            onehot_cols.append(col)
        else:
            categories = sorted(df[col].astype(str).unique().tolist())

            mapping = {
                category: index
                for index, category in enumerate(categories)
            }

            mapping["__UNKNOWN__"] = len(mapping)
            label_maps[col] = mapping

    return {
        "onehot_cols": onehot_cols,
        "label_maps": label_maps
    }


def apply_categorical_encoding(df, params):
    df = df.copy()

    for col, mapping in params["label_maps"].items():
        if col in df.columns:
            unknown_code = mapping["__UNKNOWN__"]

            df[col] = (
                df[col]
                .astype(str)
                .map(mapping)
                .fillna(unknown_code)
                .astype(int)
            )

    existing_onehot_cols = [
        col for col in params["onehot_cols"]
        if col in df.columns
    ]

    df = pd.get_dummies(
        df,
        columns=existing_onehot_cols,
        dtype=int
    )

    return df


encoding_params = fit_categorical_encoding_params(train_processed)

train_encoded = apply_categorical_encoding(
    train_processed,
    encoding_params
)

test_encoded = apply_categorical_encoding(
    test_processed,
    encoding_params
)

print("one-hot columns:")
print(encoding_params["onehot_cols"])

print("label-encoded columns:")
print(list(encoding_params["label_maps"].keys()))

print("train shape after encoding:", train_encoded.shape)
print("test shape after encoding:", test_encoded.shape)

In [ ]:
# 5) alignment and checks

def create_final_matrices(train_df, test_df):
    feature_cols = [
        col for col in train_df.columns
        if col != "target"
    ]

    X_train = train_df[feature_cols].copy()
    y_train = train_df["target"].copy()

    X_test = test_df.reindex(
        columns=feature_cols,
        fill_value=0
    ).copy()

    return X_train, y_train, X_test


X_train, y_train, X_test = create_final_matrices(
    train_encoded,
    test_encoded
)

assert "target" not in X_train.columns
assert "target" not in X_test.columns
assert "Closed Date" not in X_train.columns
assert "Closed Date" not in X_test.columns
assert X_train.shape[1] == X_test.shape[1]
assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0
assert len(X_train.select_dtypes(exclude="number").columns) == 0
assert len(X_test.select_dtypes(exclude="number").columns) == 0

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
# optional visual check

display(X_train.head())
display(y_train.head())
display(X_test.head())

In [ ]:
# The first section imports the required libraries, identifies the correct project folder, and loads train.csv and test.csv into two dataframes. 
# The original CSV files are only read, so they are not modified.

# The second section creates the target variable in the training set, assigning value 1 to requests closed within 24 hours and value 0 to all other requests. 
# It also extracts temporal features from Created Date, such as hour of the day, day of the week, month, weekend indicator, and part of the day. 
# Finally, it removes columns that should not be used for modelling, such as Closed Date, Created Date, and identifier columns.

# The third section handles missing values. Columns with too many missing values are dropped, while missing numerical values are replaced with the median calculated 
# from the training set. Missing categorical values are replaced with UNKNOWN.

# The fourth section converts categorical variables into numerical format. Columns with few categories are encoded using One-Hot Encoding, while columns with many categories 
# are encoded using Label Encoding. The encoding rules are learned only from the training set and then applied to the test set.

# The fifth section creates the final matrices used for modelling: X_train contains the training features, y_train contains the target variable, and X_test contains the test features. 
# It also checks that there are no leakage columns, missing values, or non-numerical variables, and that train and test have the same number of features.